# Black-Derman-Toy Swaption and Defaultable Bond Pricing

This notebook solves the three questions from the assignment:

1. Calibrate a 10-period Black-Derman-Toy (BDT) short-rate tree with $b=0.05$, then price a payer swaption.
2. Repeat the same swaption pricing problem with $b=0.10$.
3. Price a 10-period defaultable zero-coupon bond using a binomial short-rate tree and node-dependent hazard rates.

## Important modeling convention

For Questions 1 and 2, this notebook uses the common BDT convention:

$$
r_{i,j}=a_i e^{bj}, \qquad i=0,\ldots,9,\quad j=0,\ldots,i
$$

where $r_{i,j}$ is the one-period short rate from time $i$ to $i+1$.

Risk-neutral up/down probabilities are assumed to be:

$$
q=1-q=0.5
$$

The market zero-coupon price for maturity $T$ is:

$$
Z_0^T=\frac{100}{(1+s_T)^T}
$$

where $s_T$ is the market spot rate for maturity $T$.

## Markdown rendering note

This version uses `$...$` for inline math and `$$...$$` for display math, so formulas should render correctly in Jupyter Notebook, JupyterLab, VS Code, and Google Colab.


## Imports and Market Data

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import brentq

pd.options.display.float_format = "{:,.10f}".format

# Market spot rates from the problem
spot_rates = np.array([
    0.0300, 0.0310, 0.0320, 0.0330, 0.0340,
    0.0350, 0.0355, 0.0360, 0.0365, 0.0370
])

periods = np.arange(1, 11)
FACE = 100.0

# Market zero-coupon bond prices
market_zcb = FACE / (1.0 + spot_rates) ** periods

market_df = pd.DataFrame({
    "period": periods,
    "spot_rate": spot_rates,
    "market_zcb": market_zcb
})

market_df

,period,spot_rate,market_zcb
0,1,0.0300000000,97.0873786408
1,2,0.0310000000,94.0768287830
2,3,0.0320000000,90.9831372872
3,4,0.0330000000,87.8210679244
4,5,0.0340000000,84.6052486575
5,6,0.0350000000,81.3500644308
6,7,0.0355000000,78.3338148919
7,8,0.0360000000,75.3567147540
8,9,0.0365000000,72.4229552024
9,10,0.0370000000,69.5364373439


The problem's example says that the 6-period zero-coupon bond price should be:

$$
Z_0^6=\frac{100}{(1+0.035)^6}\approx 81.35
$$

The table above verifies the same calculation.

# Helper Functions

The BDT calibration is done sequentially. For maturity $T=i+1$, we solve for $a_i$ while keeping the previously calibrated values $a_0,\ldots,a_{i-1}$ fixed.

The zero-coupon price is computed by backward induction:

$$
V_{i,j}=\frac{0.5V_{i+1,j}+0.5V_{i+1,j+1}}{1+r_{i,j}}
$$

with terminal value $V_{T,j}=100$.

In [2]:
def zcb_price_from_tree(r_tree, maturity, face=100.0, q=0.5):
    """
    Price a zero-coupon bond by backward induction.

    Parameters
    ----------
    r_tree:
        List of arrays. r_tree[i][j] is the one-period short rate at node (i, j).
    maturity:
        Integer maturity T. Uses rates from i=0 to T-1.
    face:
        Face value paid at maturity.
    q:
        Risk-neutral probability of an up move.

    Returns
    -------
    float
        Time-0 price.
    """
    values = np.full(maturity + 1, face, dtype=float)

    for i in range(maturity - 1, -1, -1):
        new_values = np.empty(i + 1)

        for j in range(i + 1):
            expected_next = (1.0 - q) * values[j] + q * values[j + 1]
            new_values[j] = expected_next / (1.0 + r_tree[i][j])

        values = new_values

    return float(values[0])


def make_bdt_rates_for_one_period(a_i, b, i):
    """
    Build the rates at one BDT time level i:

        r_{i,j} = a_i * exp(b * j)
    """
    return np.array([a_i * np.exp(b * j) for j in range(i + 1)], dtype=float)


def calibrate_bdt_tree(spot_rates, b, face=100.0, q=0.5, tol=1e-14):
    """
    Sequentially calibrate a 10-period BDT tree.

    The calibration target is the market zero-coupon curve.
    """
    n = len(spot_rates)
    periods = np.arange(1, n + 1)
    market_zcb = face / (1.0 + spot_rates) ** periods

    a_values = np.empty(n)
    r_tree = []

    for i in range(n):
        target_price = market_zcb[i]

        def pricing_error(a_i):
            candidate_tree = r_tree + [make_bdt_rates_for_one_period(a_i, b, i)]
            model_price = zcb_price_from_tree(
                candidate_tree,
                maturity=i + 1,
                face=face,
                q=q
            )
            return model_price - target_price

        # Bond price decreases as a_i increases.
        # Start with a wide positive bracket.
        low = 1e-12
        high = 1.0

        f_low = pricing_error(low)
        f_high = pricing_error(high)

        while f_low * f_high > 0:
            high *= 2.0
            f_high = pricing_error(high)

            if high > 1_000:
                raise RuntimeError(f"Could not bracket root for period {i + 1}.")

        a_i = brentq(pricing_error, low, high, xtol=tol, rtol=tol, maxiter=1_000)

        a_values[i] = a_i
        r_tree.append(make_bdt_rates_for_one_period(a_i, b, i))

    model_zcb = np.array([
        zcb_price_from_tree(r_tree, maturity=T, face=face, q=q)
        for T in range(1, n + 1)
    ])

    calibration_df = pd.DataFrame({
        "period": periods,
        "spot_rate": spot_rates,
        "market_zcb": market_zcb,
        "model_zcb": model_zcb,
        "error": model_zcb - market_zcb,
        "a_i": a_values,
    })

    return a_values, r_tree, calibration_df


def tree_to_dataframe(r_tree):
    """
    Convert a recombining tree into a display-friendly DataFrame.
    """
    data = []

    for i, level in enumerate(r_tree):
        row = {"time_i": i}
        for j, rate in enumerate(level):
            row[f"j={j}"] = rate
        data.append(row)

    return pd.DataFrame(data)

# Questions 1 and 2: Payer Swaption Pricing

The payer swaption expires at $t=3$. If exercised, the holder enters a payer swap with payments at:

$$
t=4,5,\ldots,10
$$

The underlying swap has fixed rate:

$$
K=3.9\%
$$

For a payer swap, the holder pays fixed and receives floating. Therefore, the cash flow paid at $t=i+1$, based on the short rate prevailing during period $i$, is:

$$
N(r_{i,j}-K)
$$

The underlying swap value is computed by backward induction from $t=10$ to $t=3$:

$$
S_{i,j} = \frac{N(r_{i,j}-K) + 0.5S_{i+1,j}+0.5S_{i+1,j+1}}{1+r_{i,j}}
$$

At expiry $t=3$, the swaption payoff is:

$$
\max(S_{3,j}-0,0)
$$

because the option strike is 0.

Finally, we discount the option payoff back to time 0 through the BDT tree.

In [3]:
def price_payer_swaption(
    r_tree,
    fixed_rate=0.039,
    notional=1_000_000.0,
    expiry=3,
    maturity=10,
    q=0.5,
    option_strike=0.0
):
    """
    Price a payer swaption on the calibrated BDT tree.

    Payer swap means:
        receive floating, pay fixed

    Floating cash flow at time i+1 is based on r_{i,j}.
    """
    # Value of remaining swap at maturity after all payments have been made
    swap_next = np.zeros(maturity + 1)

    # Build underlying swap value backward from maturity-1 to expiry
    for i in range(maturity - 1, expiry - 1, -1):
        swap_current = np.empty(i + 1)

        for j in range(i + 1):
            floating_minus_fixed_cf = notional * (r_tree[i][j] - fixed_rate)
            expected_future_swap_value = (
                (1.0 - q) * swap_next[j] + q * swap_next[j + 1]
            )

            swap_current[j] = (
                floating_minus_fixed_cf + expected_future_swap_value
            ) / (1.0 + r_tree[i][j])

        swap_next = swap_current

    swap_value_at_expiry = swap_next
    swaption_payoff_at_expiry = np.maximum(
        swap_value_at_expiry - option_strike,
        0.0
    )

    # Discount option payoff from expiry back to time 0
    option_next = swaption_payoff_at_expiry

    for i in range(expiry - 1, -1, -1):
        option_current = np.empty(i + 1)

        for j in range(i + 1):
            expected_option_value = (
                (1.0 - q) * option_next[j] + q * option_next[j + 1]
            )
            option_current[j] = expected_option_value / (1.0 + r_tree[i][j])

        option_next = option_current

    return {
        "swaption_price": float(option_next[0]),
        "swap_value_at_expiry": swap_value_at_expiry,
        "swaption_payoff_at_expiry": swaption_payoff_at_expiry,
    }

## Question 1: BDT Calibration with $b=0.05$

In [4]:
b_q1 = 0.05

a_q1, r_tree_q1, calibration_q1 = calibrate_bdt_tree(
    spot_rates=spot_rates,
    b=b_q1,
    face=FACE,
    q=0.5
)

print("Maximum absolute calibration error:", calibration_q1["error"].abs().max())
calibration_q1

Maximum absolute calibration error: 2.1316282072803006e-13


,period,spot_rate,market_zcb,model_zcb,error,a_i
0,1,0.0300000000,97.0873786408,97.0873786408,-0.0000000000,0.0300000000
1,2,0.0310000000,94.0768287830,94.0768287830,-0.0000000000,0.0312017177
2,3,0.0320000000,90.9831372872,90.9831372872,-0.0000000000,0.0323263172
3,4,0.0330000000,87.8210679244,87.8210679244,0.0000000000,0.0333770580
4,5,0.0340000000,84.6052486575,84.6052486575,0.0000000000,0.0343570932
5,6,0.0350000000,81.3500644308,81.3500644308,0.0000000000,0.0352694711
6,7,0.0355000000,78.3338148919,78.3338148919,0.0000000000,0.0330953246
7,8,0.0360000000,75.3567147540,75.3567147540,0.0000000000,0.0331130643
8,9,0.0365000000,72.4229552024,72.4229552024,0.0000000000,0.0331106534
9,10,0.0370000000,69.5364373439,69.5364373439,-0.0000000000,0.0330891600


The maximum calibration error should be below $10^{-8}$, satisfying the requirement in the question.

In [5]:
print("BDT short-rate tree for Question 1, b = 0.05")
tree_to_dataframe(r_tree_q1)

BDT short-rate tree for Question 1, b = 0.05


,time_i,j=0,j=1,j=2,j=3,j=4,j=5,j=6,j=7,j=8,j=9
0,0,0.0300000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,0.0312017177,0.0328014640,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,0.0323263172,0.0339837229,0.0357261056,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,0.0333770580,0.0350883364,0.0368873539,0.0387786090,NaN,NaN,NaN,NaN,NaN,NaN
4,4,0.0343570932,0.0361186191,0.0379704603,0.0399172474,0.0419638484,NaN,NaN,NaN,NaN,NaN
5,5,0.0352694711,0.0370777756,0.0389787938,0.0409772793,0.0430782293,0.0452868974,NaN,NaN,NaN,NaN
6,6,0.0330953246,0.0347921582,0.0365759903,0.0384512814,0.0404227208,0.0424952380,0.0446740154,NaN,NaN,NaN
7,7,0.0331130643,0.0348108074,0.0365955956,0.0384718920,0.0404443880,0.0425180161,0.0446979615,0.0469896749,NaN,NaN
8,8,0.0331106534,0.0348082729,0.0365929312,0.0384690909,0.0404414434,0.0425149205,0.0446947071,0.0469862538,0.0493952905,NaN
9,9,0.0330891600,0.0347856775,0.0365691773,0.0384441191,0.0404151913,0.0424873224,0.0446656940,0.0469557531,0.0493632261,0.0518941328


In [6]:
swaption_q1 = price_payer_swaption(
    r_tree_q1,
    fixed_rate=0.039,
    notional=1_000_000.0,
    expiry=3,
    maturity=10,
    q=0.5,
    option_strike=0.0
)

print("Swap value at expiry t=3, by node:")
print(swaption_q1["swap_value_at_expiry"])

print("\nSwaption payoff at expiry t=3, by node:")
print(swaption_q1["swaption_payoff_at_expiry"])

print("\nQuestion 1 payer swaption price:")
print(swaption_q1["swaption_price"])

print("\nQuestion 1 answer rounded to nearest integer:")
print(round(swaption_q1["swaption_price"]))

Swap value at expiry t=3, by node:
[-17081.48382205  -5751.86437675   5991.18909462  18154.32233319]

Swaption payoff at expiry t=3, by node:
[    0.             0.          5991.18909462 18154.32233319]

Question 1 payer swaption price:
4102.117637556691

Question 1 answer rounded to nearest integer:
4102


## Question 2: Repeat with $b=0.10$

Now we repeat the same calibration and swaption pricing process, but with:

$$
b=0.10
$$

Because the volatility parameter changes, the full BDT tree must be recalibrated.

In [7]:
b_q2 = 0.10

a_q2, r_tree_q2, calibration_q2 = calibrate_bdt_tree(
    spot_rates=spot_rates,
    b=b_q2,
    face=FACE,
    q=0.5
)

print("Maximum absolute calibration error:", calibration_q2["error"].abs().max())
calibration_q2

Maximum absolute calibration error: 1.7053025658242404e-13


,period,spot_rate,market_zcb,model_zcb,error,a_i
0,1,0.0300000000,97.0873786408,97.0873786408,-0.0000000000,0.0300000000
1,2,0.0310000000,94.0768287830,94.0768287830,-0.0000000000,0.0304046076
2,3,0.0320000000,90.9831372872,90.9831372872,-0.0000000000,0.0306977371
3,4,0.0330000000,87.8210679244,87.8210679244,-0.0000000000,0.0308900685
4,5,0.0340000000,84.6052486575,84.6052486575,0.0000000000,0.0309914918
5,6,0.0350000000,81.3500644308,81.3500644308,-0.0000000000,0.0310111542
6,7,0.0355000000,78.3338148919,78.3338148919,0.0000000000,0.0283663431
7,8,0.0360000000,75.3567147540,75.3567147540,0.0000000000,0.0276687596
8,9,0.0365000000,72.4229552024,72.4229552024,0.0000000000,0.0269742348
9,10,0.0370000000,69.5364373439,69.5364373439,0.0000000000,0.0262843472


In [8]:
print("BDT short-rate tree for Question 2, b = 0.10")
tree_to_dataframe(r_tree_q2)

BDT short-rate tree for Question 2, b = 0.10


,time_i,j=0,j=1,j=2,j=3,j=4,j=5,j=6,j=7,j=8,j=9
0,0,0.0300000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,0.0304046076,0.0336022881,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,0.0306977371,0.0339262462,0.0374943007,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,0.0308900685,0.0341388054,0.0377292149,0.0416972311,NaN,NaN,NaN,NaN,NaN,NaN
4,4,0.0309914918,0.0342508955,0.0378530936,0.0418341382,0.0462338729,NaN,NaN,NaN,NaN,NaN
5,5,0.0310111542,0.0342726258,0.0378771093,0.0418606796,0.0462632057,0.0511287496,NaN,NaN,NaN,NaN
6,6,0.0283663431,0.0313496574,0.0346467296,0.0382905580,0.0423176111,0.0467681932,0.0516868470,NaN,NaN,NaN
7,7,0.0276687596,0.0305787084,0.0337946993,0.0373489188,0.0412769389,0.0456180724,0.0504157670,0.0557180395,NaN,NaN
8,8,0.0269742348,0.0298111398,0.0329464048,0.0364114084,0.0402408297,0.0444729947,0.0491502604,0.0543194384,0.0600322636,NaN
9,9,0.0262843472,0.0290486961,0.0321037741,0.0354801575,0.0392116383,0.0433355622,0.0478932031,0.0529301753,0.0584968904,0.0646490620


In [9]:
swaption_q2 = price_payer_swaption(
    r_tree_q2,
    fixed_rate=0.039,
    notional=1_000_000.0,
    expiry=3,
    maturity=10,
    q=0.5,
    option_strike=0.0
)

print("Swap value at expiry t=3, by node:")
print(swaption_q2["swap_value_at_expiry"])

print("\nSwaption payoff at expiry t=3, by node:")
print(swaption_q2["swaption_payoff_at_expiry"])

print("\nQuestion 2 payer swaption price:")
print(swaption_q2["swaption_price"])

print("\nQuestion 2 answer rounded to nearest integer:")
print(round(swaption_q2["swaption_price"]))

Swap value at expiry t=3, by node:
[-33681.54684124 -11871.49568852  11573.58574125  36706.37227398]

Swaption payoff at expiry t=3, by node:
[    0.             0.         11573.58574125 36706.37227398]

Question 2 payer swaption price:
8096.569715697246

Question 2 answer rounded to nearest integer:
8097


# Question 3: Defaultable Zero-Coupon Bond

For Question 3, no calibration is needed.

We construct a 10-period binomial short-rate tree:

$$
r_{i,j}=r_{0,0}u^jd^{i-j}
$$

with:

$$
r_{0,0}=5\%, \qquad u=1.1, \qquad d=0.9, \qquad q=0.5
$$

The one-step hazard rate is:

$$
h_{i,j}=ab^{j-\frac{i}{2}}
$$

with:

$$
a=0.01, \qquad b=1.01
$$

The terminal payoff of the zero-coupon bond is:

$$
F=100
$$

and recovery is:

$$
RF=0.20 \times 100 = 20
$$

At each node, the defaultable bond value is computed as:

$$
V_{i,j}=
\frac{
(1-h_{i,j})\left[0.5V_{i+1,j}+0.5V_{i+1,j+1}\right]
+h_{i,j}(RF)
}{
1+r_{i,j}
}
$$

The first term represents survival. The second term represents default and recovery.

In [10]:
def build_q3_short_rate_tree(n=10, r0=0.05, u=1.1, d=0.9):
    """
    Build the binomial short-rate tree for Question 3:

        r_{i,j} = r0 * u^j * d^(i-j)
    """
    r_tree = []

    for i in range(n):
        level = np.array([
            r0 * (u ** j) * (d ** (i - j))
            for j in range(i + 1)
        ], dtype=float)
        r_tree.append(level)

    return r_tree


def build_hazard_tree(n=10, a=0.01, b=1.01):
    """
    Build the hazard-rate tree:

        h_{i,j} = a * b^(j - i/2)
    """
    h_tree = []

    for i in range(n):
        level = np.array([
            a * (b ** (j - i / 2.0))
            for j in range(i + 1)
        ], dtype=float)
        h_tree.append(level)

    return h_tree


def price_defaultable_zcb(
    n=10,
    r0=0.05,
    u=1.1,
    d=0.9,
    q=0.5,
    hazard_a=0.01,
    hazard_b=1.01,
    face=100.0,
    recovery_rate=0.20
):
    """
    Price the defaultable zero-coupon bond by backward induction.
    """
    r_tree = build_q3_short_rate_tree(n=n, r0=r0, u=u, d=d)
    h_tree = build_hazard_tree(n=n, a=hazard_a, b=hazard_b)

    values = np.full(n + 1, face, dtype=float)
    recovery_amount = recovery_rate * face

    for i in range(n - 1, -1, -1):
        current_values = np.empty(i + 1)

        for j in range(i + 1):
            survival_continuation_value = (
                (1.0 - q) * values[j] + q * values[j + 1]
            )

            current_values[j] = (
                (1.0 - h_tree[i][j]) * survival_continuation_value
                + h_tree[i][j] * recovery_amount
            ) / (1.0 + r_tree[i][j])

        values = current_values

    return {
        "price": float(values[0]),
        "r_tree": r_tree,
        "h_tree": h_tree,
    }

In [11]:
q3_result = price_defaultable_zcb(
    n=10,
    r0=0.05,
    u=1.1,
    d=0.9,
    q=0.5,
    hazard_a=0.01,
    hazard_b=1.01,
    face=100.0,
    recovery_rate=0.20
)

print("Question 3 defaultable zero-coupon bond price:")
print(q3_result["price"])

print("\nQuestion 3 answer rounded to two decimal places:")
print(round(q3_result["price"], 2))

Question 3 defaultable zero-coupon bond price:
57.216858239429015

Question 3 answer rounded to two decimal places:
57.22


In [12]:
print("Question 3 short-rate tree:")
tree_to_dataframe(q3_result["r_tree"])

Question 3 short-rate tree:


,time_i,j=0,j=1,j=2,j=3,j=4,j=5,j=6,j=7,j=8,j=9
0,0,0.0500000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,0.0450000000,0.0550000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,0.0405000000,0.0495000000,0.0605000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,0.0364500000,0.0445500000,0.0544500000,0.0665500000,NaN,NaN,NaN,NaN,NaN,NaN
4,4,0.0328050000,0.0400950000,0.0490050000,0.0598950000,0.0732050000,NaN,NaN,NaN,NaN,NaN
5,5,0.0295245000,0.0360855000,0.0441045000,0.0539055000,0.0658845000,0.0805255000,NaN,NaN,NaN,NaN
6,6,0.0265720500,0.0324769500,0.0396940500,0.0485149500,0.0592960500,0.0724729500,0.0885780500,NaN,NaN,NaN
7,7,0.0239148450,0.0292292550,0.0357246450,0.0436634550,0.0533664450,0.0652256550,0.0797202450,0.0974358550,NaN,NaN
8,8,0.0215233605,0.0263063295,0.0321521805,0.0392971095,0.0480298005,0.0587030895,0.0717482205,0.0876922695,0.1071794405,NaN
9,9,0.0193710245,0.0236756966,0.0289369625,0.0353673986,0.0432268205,0.0528327806,0.0645733985,0.0789230426,0.0964614965,0.1178973846


In [13]:
print("Question 3 hazard-rate tree:")
tree_to_dataframe(q3_result["h_tree"])

Question 3 hazard-rate tree:


,time_i,j=0,j=1,j=2,j=3,j=4,j=5,j=6,j=7,j=8,j=9
0,0,0.0100000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,0.0099503719,0.0100498756,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,0.0099009901,0.0100000000,0.0101000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,0.0098518534,0.0099503719,0.0100498756,0.0101503744,NaN,NaN,NaN,NaN,NaN,NaN
4,4,0.0098029605,0.0099009901,0.0100000000,0.0101000000,0.0102010000,NaN,NaN,NaN,NaN,NaN
5,5,0.0097543103,0.0098518534,0.0099503719,0.0100498756,0.0101503744,0.0102518781,NaN,NaN,NaN,NaN
6,6,0.0097059015,0.0098029605,0.0099009901,0.0100000000,0.0101000000,0.0102010000,0.0103030100,NaN,NaN,NaN
7,7,0.0096577329,0.0097543103,0.0098518534,0.0099503719,0.0100498756,0.0101503744,0.0102518781,0.0103543969,NaN,NaN
8,8,0.0096098034,0.0097059015,0.0098029605,0.0099009901,0.0100000000,0.0101000000,0.0102010000,0.0103030100,0.0104060401,NaN
9,9,0.0095621118,0.0096577329,0.0097543103,0.0098518534,0.0099503719,0.0100498756,0.0101503744,0.0102518781,0.0103543969,0.0104579409


# Final Answers

Using the convention $r_{i,j}=a_i e^{bj}$ for the BDT model:

| Question | Result |
|---|---:|
| Q1 payer swaption price, $b=0.05$ | 4,102 |
| Q2 payer swaption price, $b=0.10$ | 8,097 |
| Q3 defaultable zero-coupon bond price | 57.22 |

These are the rounded submission values produced by the notebook.